# 0603 — ETL: pienza.db to BigQuery

This notebook is the bridge between the SQLite database built in `0211_ETL_Big_Bang_pienzadb.ipynb` and the BigQuery dataset that powers the live Streamlit Observatory. It migrates every table, rebuilds every view natively in BigQuery SQL, audits relational integrity, certifies row/column parity against the SQLite source, and backs up the raw database file to GCS.

```mermaid
flowchart TD
    A["pienza.db (SQLite)"] --> B["Phase 1\nSetup + BigQuery client"]
    B --> C["Phase 2\nTable migration"]
    C --> D["Phase 3\nRelational audit"]
    D --> E["Phase 4\nView reconstruction"]
    E --> F["Phase 5\nParity audit"]
    F --> G["Phase 6\nGCS backup"]

    classDef default stroke:#21918c,stroke-width:2px;
    linkStyle default stroke:#21918c,stroke-width:2px;
```

## Phase 1 — Setup and connectivity

Connects to the local SQLite database and configures the BigQuery client and destination dataset.

#### 1.1 — Local database connection

Locates the local pienza.db file and prepares the SQLAlchemy engine, plus the shared visual theme.

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import sqlite3

pd.options.mode.chained_assignment = None

DB_PATH = '/workspaces/pienza/data/pienza.db'

print(f"Looking for the master DB at: {DB_PATH}")

if not os.path.exists(DB_PATH):
    print(f"CRITICAL: Database not found at {DB_PATH}")
else:
    print("Database found. Engine ready.")
    db_engine = create_engine(f'sqlite:///{DB_PATH}')

PIENZA_PURPLE, PIENZA_TEAL, PIENZA_GREY, PIENZA_TEXT = '#440154', '#21918c', '#FAFAFA', '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': PIENZA_GREY,
    'axes.facecolor': PIENZA_GREY,
    'text.color': PIENZA_TEXT,
    'axes.titlecolor': PIENZA_PURPLE,
    'axes.titleweight': 'bold',
    'figure.titlesize': 20
})

print("Visual identity loaded.")

Looking for the master DB at: /workspaces/pienza/data/pienza.db
Database found. Engine ready.
Visual identity loaded.


#### 1.2 — BigQuery client and dataset

Authenticates via service account and creates the destination BigQuery dataset if it doesn't already exist.

In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

SA_PATH = "/workspaces/pienza/secrets/service-account.json"
PROJECT_ID = '645009831643'
DATASET_ID = 'pienza_mini'

if os.path.exists(SA_PATH):
    print(f"Loading credentials from: {os.path.basename(SA_PATH)}")
    credentials = service_account.Credentials.from_service_account_file(SA_PATH)
    client = bigquery.Client(credentials=credentials, project=PROJECT_ID)
    print(f"BigQuery client active: {PROJECT_ID}")
else:
    print(f"ERROR: Service account not found at {SA_PATH}")

def prepare_dataset():
    dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
    try:
        client.get_dataset(dataset_ref)
        print(f"Dataset '{DATASET_ID}' already exists. Ready to receive data.")
    except Exception:
        print(f"Dataset '{DATASET_ID}' not found. Creating it...")
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        client.create_dataset(dataset, timeout=30)
        print(f"Dataset '{DATASET_ID}' created successfully.")

prepare_dataset()

Loading credentials from: service-account.json
BigQuery client active: 645009831643
Dataset 'pienza_mini' already exists. Ready to receive data.


## Phase 2 — Migration

Migrates every SQLite table into BigQuery as a native table.

#### 2.1 — Table-by-table migration

Iterates through all SQLite tables, normalizes column names, and loads each one into BigQuery with WRITE_TRUNCATE.

In [3]:
def complete_migration():
    print(f"Starting migration of {os.path.basename(DB_PATH)} to BigQuery...")
    print(f"Destination: {PROJECT_ID}.{DATASET_ID}")
    print("-" * 65)

    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
        tables = [row[0] for row in cursor.fetchall()]
        print(f"Detected {len(tables)} tables to migrate.")
    except Exception as e:
        print(f"ERROR ACCESSING SQLITE: {e}")
        return

    # WRITE_TRUNCATE ensures the table is fully overwritten with the master version if it exists
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

    for table in tables:
        try:
            print(f"   Migrating: {table}...")

            df = pd.read_sql_query(f"SELECT * FROM `{table}`", conn)

            # BigQuery does not accept spaces, dots, or dashes in column names
            df.columns = [c.lower().replace(' ', '_').replace('-', '_').replace('.', '_') for c in df.columns]

            table_id = f"{PROJECT_ID}.{DATASET_ID}.{table}"

            job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
            job.result()

            print(f"   {table} ({len(df):,} rows) is now native in the cloud.")

        except Exception as e:
            print(f"   ERROR on table {table}: {e}")

    conn.close()
    print("-" * 65)
    print("\nMigration complete. The star schema now lives on Google infrastructure.")

complete_migration()

Starting migration of pienza.db to BigQuery...
Destination: 645009831643.pienza_mini
-----------------------------------------------------------------
Detected 18 tables to migrate.
   Migrating: driver_state_at_request...
   driver_state_at_request (2 rows) is now native in the cloud.
   Migrating: heuristic_flag...
   heuristic_flag (8 rows) is now native in the cloud.
   Migrating: interpolation_quality...
   interpolation_quality (5 rows) is now native in the cloud.
   Migrating: offer_action...
   offer_action (2 rows) is now native in the cloud.
   Migrating: outcome...
   outcome (5 rows) is now native in the cloud.
   Migrating: post_offer_status...
   post_offer_status (2 rows) is now native in the cloud.
   Migrating: product_category...
   product_category (7 rows) is now native in the cloud.
   Migrating: reason_primary...
   reason_primary (7 rows) is now native in the cloud.
   Migrating: record_status...
   record_status (2 rows) is now native in the cloud.
   Migrating:

## Phase 3 — Relational integrity audit

Validates that key foreign-key relationships survived the migration intact.

#### 3.1 — Orphan and connectivity checks

Runs a handful of orphan/connectivity checks across offers, product_category, offer_action, engineered_features, and heuristic flags.

In [4]:
def execute_relational_audit():
    print(f"Starting relational audit on dataset: {DATASET_ID}...\n")

    audits = [
        ("1. TOTAL OFFERS",
         f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_ID}.offers`"),

        ("2. PRODUCT INTEGRITY (Orphans)",
         f"""SELECT COUNT(o.offer_id) as orphans
             FROM `{PROJECT_ID}.{DATASET_ID}.offers` o
             LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.product_category` p
             ON o.product_category_fk = p.product_category_id
             WHERE p.product_category_id IS NULL"""),

        ("3. OFFERS <-> FEATURES CONNECTIVITY",
         f"""SELECT COUNT(o.offer_id) as total_links
             FROM `{PROJECT_ID}.{DATASET_ID}.offers` o
             INNER JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef
             ON o.offer_id = ef.offer_id_fk"""),

        ("4. ACTION INTEGRITY (Orphans)",
         f"""SELECT COUNT(o.offer_id) as orphans
             FROM `{PROJECT_ID}.{DATASET_ID}.offers` o
             LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.offer_action` oa
             ON o.offer_action_fk = oa.offer_action_id
             WHERE oa.offer_action_id IS NULL"""),

        ("5. HEURISTIC FLAGS BRIDGE (Linkage)",
         f"""SELECT COUNT(*) as total_flags
             FROM `{PROJECT_ID}.{DATASET_ID}.heuristic_flag_offers` hfo
             JOIN `{PROJECT_ID}.{DATASET_ID}.heuristic_flag` hf
             ON hfo.heuristic_flag_heuristic_flag_id = hf.heuristic_flag_id""")
    ]

    for title, query in audits:
        try:
            res = client.query(query).to_dataframe()
            val = res.iloc[0, 0]

            status = "OK"
            if "Orphans" in title and val > 0:
                status = f"ERROR: {val} orphans found!"
            if "TOTAL OFFERS" in title:
                status = f"{val} records"
            if "total_links" in title and val < 4000:
                status = f"ALERT: Only {val} connections found."

            print(f"{title:<40} {status}")
        except Exception as e:
            print(f"Error on test '{title}': {e}")

    print("\nAUDIT FINISHED.")

execute_relational_audit()

Starting relational audit on dataset: pienza_mini...

1. TOTAL OFFERS                          4765 records
2. PRODUCT INTEGRITY (Orphans)           OK
3. OFFERS <-> FEATURES CONNECTIVITY      OK
4. ACTION INTEGRITY (Orphans)            OK
5. HEURISTIC FLAGS BRIDGE (Linkage)      OK

AUDIT FINISHED.


## Phase 4 — View reconstruction

Rebuilds every SQLite view natively in BigQuery SQL (SAFE_CAST/TIMESTAMP_DIFF/SAFE_DIVIDE replacing SQLite's julianday-based logic).

#### 4.1 — Extract original view definitions

Reads the original view SQL straight from SQLite, as a reference for the BigQuery rewrite below.

In [5]:
def extract_view_logic():
    print(f"Extracting view definitions from {DB_PATH}...\n")

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='view';")
    views = cursor.fetchall()

    if not views:
        print("No views found in the database.")
    else:
        for name, sql in views:
            print(f"--- VIEW: {name} ---")
            print(sql)
            print("\n" + "="*50 + "\n")

    conn.close()

extract_view_logic()

Extracting view definitions from /workspaces/pienza/data/pienza.db...

--- VIEW: v_trip_funnel_metrics ---
CREATE VIEW v_trip_funnel_metrics AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,
    (julianday(p.t2_arrived) - julianday(p.t1_accepted)) * 8640

#### 4.2 — Forge v_trip_funnel_metrics

Pivots the long-format trip_events log into per-trip T0-T4 durations.

In [6]:
view_name = "v_trip_funnel_metrics"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        -- SAFE_CAST used throughout for cross-system timestamp integrity
        MAX(CASE WHEN event_types_id_fk = 1 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(realized_fare AS FLOAT64) END) AS realized_fare
    FROM
        `{PROJECT_ID}.{DATASET_ID}.trip_events`
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,

    -- julianday deltas (SQLite) become TIMESTAMP_DIFF (BigQuery)
    TIMESTAMP_DIFF(p.t2_arrived, p.t1_accepted, SECOND) AS duration_to_pickup_sec,
    TIMESTAMP_DIFF(p.t3_started, p.t2_arrived, SECOND) AS duration_waiting_sec,
    TIMESTAMP_DIFF(p.t4_completed, p.t3_started, SECOND) AS duration_trip_sec,
    TIMESTAMP_DIFF(p.t4_completed, p.t1_accepted, SECOND) AS duration_total_engagement_sec,

    p.t0_looking,
    p.t1_accepted,
    p.t2_arrived,
    p.t3_started,
    p.t4_completed
FROM
    PivotedEvents p
"""

try:
    print(f"Building view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} is now native in BigQuery.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building view: v_trip_funnel_metrics...
SUCCESS: v_trip_funnel_metrics is now native in BigQuery.


#### 4.3 — Forge v_trip_funnel_wide

Wide-format version of the same event pivot, keyed by trip_id_legacy.

In [7]:
view_name = "v_trip_funnel_wide"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    trip_id_legacy,
    MAX(offer_id_fk) AS offer_id_fk,
    MAX(CASE WHEN event_types_id_fk = 1 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(realized_fare AS FLOAT64) END) AS realized_fare
FROM `{PROJECT_ID}.{DATASET_ID}.trip_events`
GROUP BY trip_id_legacy
"""

try:
    print(f"Building view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} is now native in BigQuery.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building view: v_trip_funnel_wide...
SUCCESS: v_trip_funnel_wide is now native in BigQuery.


#### 4.4 — Forge v_trip_final_kpis

Computes spread percentage and earnings-per-hour KPIs on top of v_trip_funnel_wide.

In [8]:
view_name = "v_trip_final_kpis"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,

    TIMESTAMP_DIFF(v.t2_timestamp, v.t1_timestamp, SECOND) AS duration_to_pickup_sec,
    TIMESTAMP_DIFF(v.t3_timestamp, v.t2_timestamp, SECOND) AS duration_waiting_sec,
    TIMESTAMP_DIFF(v.t4_timestamp, v.t3_timestamp, SECOND) AS duration_trip_sec,
    TIMESTAMP_DIFF(v.t4_timestamp, v.t1_timestamp, SECOND) AS total_engagement_duration_sec,

    v.upfront_fare,
    v.realized_fare,

    SAFE_DIVIDE(v.realized_fare, v.upfront_fare) AS spread_percentage,

    SAFE_DIVIDE(
        v.realized_fare,
        (TIMESTAMP_DIFF(v.t4_timestamp, v.t3_timestamp, SECOND) / 3600.0)
    ) AS eph_on_ride,

    SAFE_DIVIDE(
        v.realized_fare,
        (TIMESTAMP_DIFF(v.t4_timestamp, v.t1_timestamp, SECOND) / 3600.0)
    ) AS eph_total_time

FROM `{PROJECT_ID}.{DATASET_ID}.v_trip_funnel_wide` v
"""

try:
    print(f"Building view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} is now native in BigQuery.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building view: v_trip_final_kpis...
SUCCESS: v_trip_final_kpis is now native in BigQuery.


#### 4.5 — Forge v_mission_dossier

Links each trip back to its originating offer_id and merges in the final KPIs.

In [9]:
view_name = "v_mission_dossier"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
WITH OfferLink AS (
    SELECT
        trip_id_legacy,
        MAX(offer_id_fk) AS offer_id
    FROM
        `{PROJECT_ID}.{DATASET_ID}.trip_events`
    GROUP BY
        trip_id_legacy
)
SELECT
    ol.offer_id,
    kpi.*
FROM
    `{PROJECT_ID}.{DATASET_ID}.v_trip_final_kpis` AS kpi
LEFT JOIN
    OfferLink AS ol ON kpi.trip_id_legacy = ol.trip_id_legacy
"""

try:
    print(f"Building view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} is now native in BigQuery.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building view: v_mission_dossier...
SUCCESS: v_mission_dossier is now native in BigQuery.


#### 4.6 — Forge v_broche_fks

Full data-lineage audit view, joining OCR through to earnings via offer_id/lifetime_trips_id.

In [10]:
view_name = "v_broche_fks"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    r.ocr_id                  AS raw_ocr_id,

    o.offer_id                AS hub_offer_id,
    o.session_fk              AS hub_session_fk,
    o.ocr_fk                  AS hub_ocr_fk,

    ef.feature_id             AS feat_id,
    ef.offer_id_fk            AS feat_offer_id_fk,

    te.event_id               AS event_id,
    te.offer_id_fk            AS event_offer_id_fk,

    lt.lifetime_trips_id      AS lt_id,
    lt.offer_id_fk            AS lt_offer_id_fk,

    ae.activity_earnings_id   AS ae_id,
    ae.offer_id_fk            AS ae_offer_id_fk,
    ae.lifetime_trips_fk      AS ae_lt_fk

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.raw_offers_ocr` r        ON o.ocr_fk = r.ocr_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.trip_events` te          ON o.offer_id = te.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.lifetime_trips` lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.activity_earnings` ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk
"""

try:
    print(f"Building view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} is now native in BigQuery.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building view: v_broche_fks...
SUCCESS: v_broche_fks is now native in BigQuery.


#### 4.7 — Forge v_offers_human

Human-readable master view, resolving every offer_id foreign key into its dimension-table label.

In [11]:
view_name = "v_offers_human"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    -- 1. IDENTITY & TIME
    o.offer_id,
    o.session_fk,
    o.offer_timestamp,

    -- 2. CORE METRICS
    o.upfront_fare,
    o.time_to_pickup_sec,
    o.dist_to_pickup_km,
    o.est_trip_time_sec,
    o.est_trip_dist_km,

    -- 3. HUMAN-READABLE LABELS
    pc.category_name                  AS str_product,
    oa.offer_action_description       AS str_action,
    rp.reason_primary_description     AS str_reason,
    ds.driver_state_at_request_description AS str_driver_state,
    pos.post_offer_status_description AS str_post_status,
    out_dim.outcome_description       AS str_outcome,
    iq.interpolation_quality_description AS str_interp_quality,
    rs.record_status_description      AS str_record_status,

    -- 4. GEOSPATIAL CONTEXT
    o.pickup_address,
    o.dropoff_address,
    o.pickup_lat, o.pickup_lon,
    o.dropoff_lat, o.dropoff_lon,

    -- 5. FLAGS & NOTES
    o.is_surge, o.surge_amount,
    o.is_turbo_plus, o.turbo_plus_amount,
    o.is_reservation, o.reservation_amount,
    o.special_note_raw,

    -- 6. ENGINEERED FEATURES (ERD-aligned)
    ef.pickup_ambiguity,
    ef.dropoff_ambiguity,
    ef.traffic_index_base_120,
    ef.time_since_last_offer,
    ef.offer_density_60sec,
    ef.cycle_avg_dtp_km,
    ef.cycle_rolling_avg_spread,
    ef.total_accumulated_deadhead_sec,
    ef.cycle_cumulative_net_earnings,
    ef.eph_direct,
    ef.eph_operational,
    ef.is_operational_downgrade,

    ef.eph_realized_ML,
    ef.eph_complete_ML,
    ef.is_spread_downgrade_ML,
    ef.is_total_cycle_downgrade_ML,

    ef.home_vector_alignment_score,
    ef.day_of_week,
    ef.time_of_day_block,
    ef.day_type

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.product_category` pc        ON o.product_category_fk = pc.product_category_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.offer_action` oa            ON o.offer_action_fk = oa.offer_action_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.reason_primary` rp          ON o.reason_primary_fk = rp.reason_primary_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.driver_state_at_request` ds ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.post_offer_status` pos      ON o.post_offer_status_fk = pos.post_offer_status_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.outcome` out_dim            ON o.outcome_fk = out_dim.outcome_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.interpolation_quality` iq   ON o.interpolation_quality_fk = iq.interpolation_quality_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.record_status` rs           ON o.record_status_fk = rs.record_status_id

    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef     ON o.offer_id = ef.offer_id_fk
"""

try:
    print(f"Building view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} is now native in BigQuery.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building view: v_offers_human...
SUCCESS: v_offers_human is now native in BigQuery.


#### 4.8 — Rebuild v_lifecycle_audit

Full-parity rebuild of the lifecycle audit view, restoring columns that had drifted from the ERD, and refreshing the dependent accepted-only view.

In [12]:
view_name = "v_lifecycle_audit"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
WITH TripEventsPivot AS (
    SELECT
        offer_id_fk,
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 2 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) as t1_timestamp,
        MAX(CASE WHEN event_types_id_fk = 4 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) as t3_timestamp,
        MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) as t4_timestamp,
        MAX(upfront_fare) as te_upfront_fare,
        MAX(SAFE_CAST(realized_fare AS FLOAT64)) as te_realized_fare
    FROM `{PROJECT_ID}.{DATASET_ID}.trip_events`
    GROUP BY offer_id_fk, trip_id_legacy
),
HistoryStats AS (
    SELECT
        lt.offer_id_fk,
        lt.lifetime_trips_id,
        lt.original_fare,
        ae.net_earning,
        SUM(lt.original_fare) OVER (ORDER BY SAFE_CAST(lt.request_timestamp AS TIMESTAMP) ROWS UNBOUNDED PRECEDING) as cum_uber_earnings,
        SUM(ae.net_earning) OVER (ORDER BY SAFE_CAST(lt.request_timestamp AS TIMESTAMP) ROWS UNBOUNDED PRECEDING) as cum_net_earnings,
        AVG(SAFE_DIVIDE(ae.net_earning, lt.original_fare)) OVER (ORDER BY SAFE_CAST(lt.request_timestamp AS TIMESTAMP) ROWS UNBOUNDED PRECEDING) as rolling_avg_net_take_rate
    FROM `{PROJECT_ID}.{DATASET_ID}.lifetime_trips` lt
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.activity_earnings` ae ON lt.lifetime_trips_id = ae.lifetime_trips_fk
)
SELECT
    -- 1. IDENTIFIERS
    o.offer_id, o.session_fk, te.trip_id_legacy,

    -- 2. TIME CONSISTENCY
    r.time_taken                AS ocr_raw_time,
    SAFE_CAST(o.offer_timestamp AS TIMESTAMP) AS clean_timestamp,
    ef.day_of_week, ef.time_of_day_block,

    -- 3. PRODUCT CONSISTENCY
    r.ride_type                 AS ocr_product,
    pc.category_name            AS internal_product,
    ae.product_category         AS bank_product,
    lt.global_product_name      AS official_product,

    -- 4. SPATIAL CONSISTENCY - PICKUPS
    r.pickup_address            AS ocr_pickup,
    o.pickup_address            AS clean_pickup,
    ef.pickup_ambiguity,

    -- 5. SPATIAL CONSISTENCY - DROPOFFS
    r.dropoff_address           AS ocr_dropoff,
    o.dropoff_address           AS clean_dropoff,
    ef.dropoff_ambiguity,

    -- 6. FINANCIAL CONSISTENCY
    lt.original_fare            AS uber_original_fare,
    r.upfront_fare              AS ocr_upfront,
    o.upfront_fare              AS clean_upfront,
    te.te_upfront_fare          AS events_upfront,
    te.te_realized_fare         AS events_realized,
    ae.net_earning              AS bank_net_earning,

    -- 7. TEMPORAL AUDIT - BASE TIMESTAMPS & DELTAS
    te.t1_timestamp             AS gts_t1_accepted,
    SAFE_CAST(lt.request_timestamp AS TIMESTAMP) AS uber_request,
    TIMESTAMP_DIFF(te.t1_timestamp, SAFE_CAST(lt.request_timestamp AS TIMESTAMP), SECOND) AS delta_accept_sec,

    te.t3_timestamp             AS gts_t3_started,
    SAFE_CAST(lt.pickup_timestamp AS TIMESTAMP) AS uber_pickup,
    TIMESTAMP_DIFF(te.t3_timestamp, SAFE_CAST(lt.pickup_timestamp AS TIMESTAMP), SECOND) AS delta_start_sec,

    te.t4_timestamp             AS gts_t4_completed,
    SAFE_CAST(lt.dropoff_timestamp AS TIMESTAMP) AS uber_dropoff,
    TIMESTAMP_DIFF(te.t4_timestamp, SAFE_CAST(lt.dropoff_timestamp AS TIMESTAMP), SECOND) AS delta_end_sec,

    -- 8. HISTORICAL CONTEXT
    hist.cum_uber_earnings, hist.cum_net_earnings, hist.rolling_avg_net_take_rate

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.raw_offers_ocr` r        ON o.ocr_fk = r.ocr_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.product_category` pc     ON o.product_category_fk = pc.product_category_id
    LEFT JOIN TripEventsPivot te      ON o.offer_id = te.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.lifetime_trips` lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.activity_earnings` ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk
    LEFT JOIN HistoryStats hist       ON o.offer_id = hist.offer_id_fk
"""

try:
    print(f"Rebuilding view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} now has full column parity with the ERD.")

    accepted_view_id = f"{PROJECT_ID}.{DATASET_ID}.v_lifecycle_audit_accepted"
    accepted_sql = f"CREATE OR REPLACE VIEW `{accepted_view_id}` AS SELECT * FROM `{view_id}` WHERE trip_id_legacy IS NOT NULL"
    client.query(accepted_sql).result()
    print("SUCCESS: v_lifecycle_audit_accepted updated automatically.")

except Exception as e:
    print(f"FAILURE in rebuild: {e}")

Rebuilding view: v_lifecycle_audit...
SUCCESS: v_lifecycle_audit now has full column parity with the ERD.
SUCCESS: v_lifecycle_audit_accepted updated automatically.


#### 4.9 — Forge v_lifecycle_audit_accepted

Filters the lifecycle audit down to completed missions only.

In [13]:
view_name = "v_lifecycle_audit_accepted"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    *
FROM
    `{PROJECT_ID}.{DATASET_ID}.v_lifecycle_audit`
WHERE
    trip_id_legacy IS NOT NULL -- Filter to completed missions only
ORDER BY
    clean_timestamp DESC
"""

try:
    print(f"Building final view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} activated.")
    print("\nView migration complete.")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building final view: v_lifecycle_audit_accepted...
SUCCESS: v_lifecycle_audit_accepted activated.

View migration complete.


#### 4.10 — Forge v_ML_Supervised

The definitive flat table for ML: offers plus engineered_features, silver_palette, and heuristic-flag context, with full structural parity.

In [14]:
view_name = "v_ML_Supervised"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    -- 1. BASE: table 'offers' (o)
    o.offer_id,
    o.session_fk,
    o.ocr_fk,
    o.image_content_hash,
    o.offer_timestamp,
    EXTRACT(HOUR FROM CAST(o.offer_timestamp AS TIMESTAMP)) AS hour_of_day,
    o.upfront_fare,
    o.time_to_pickup_sec,
    o.dist_to_pickup_km,
    o.est_trip_time_sec,
    o.est_trip_dist_km,
    o.pickup_address,
    o.dropoff_address,
    o.pickup_lat,
    o.pickup_lon,
    o.dropoff_lat,
    o.dropoff_lon,
    o.is_surge,
    o.surge_amount,
    o.is_turbo_plus,
    o.turbo_plus_amount,
    o.is_reservation,
    o.reservation_amount,
    o.is_priority,
    o.priority_amount,
    o.is_exclusive,
    o.is_vip,
    o.is_identity_verified,
    o.is_long_trip,
    o.is_multiple_destinations,
    o.is_teens,
    o.rider_star_rating,
    o.rider_trip_count,
    o.time_in_session_sec,
    o.session_progress_ratio,
    o.inferred_agent_lat,
    o.inferred_agent_lon,
    o.inferred_agent_bearing,
    o.inferred_agent_speed_mps,
    o.is_imputed,
    o.special_note_raw,
    o.comment_1,
    o.comment_2,
    o.product_category_fk,
    o.offer_action_fk,
    o.reason_primary_fk,
    o.post_offer_status_fk,
    o.driver_state_at_request_fk,
    o.outcome_fk,
    o.interpolation_quality_fk,
    o.record_status_fk,

    -- 2. INTELLIGENCE: table 'engineered_features' (ef)
    ef.feature_id,
    ef.traffic_index_base_120,
    ef.time_since_last_offer,
    ef.offer_density_10sec,
    ef.offer_density_30sec,
    ef.offer_density_60sec,
    ef.offer_density_180sec,
    ef.consecutive_rejects,
    ef.cycle_avg_dtp_km,
    ef.cycle_std_dtp_km,
    ef.cycle_ttp_dtp_ratio,
    ef.dispatch_lead_time_sec,
    ef.cycle_rolling_avg_spread,
    ef.total_accumulated_deadhead_sec,
    ef.cycle_cumulative_net_earnings,
    ef.eph_direct,
    ef.eph_direct_index,
    ef.eph_direct_label,
    ef.eph_operational,
    ef.eph_operational_index,
    ef.eph_operational_label,
    ef.is_operational_downgrade,
    ef.eph_realized_ML,
    ef.eph_realized_index_ML,
    ef.eph_realized_label_ML,
    ef.is_spread_downgrade_ML,
    ef.eph_complete_ML,
    ef.eph_complete_index_ML,
    ef.eph_complete_label_ML,
    ef.is_total_cycle_downgrade_ML,
    ef.eph_realized_EDA,
    ef.eph_realized_index_EDA,
    ef.eph_realized_label_EDA,
    ef.is_spread_downgrade_EDA,
    ef.eph_complete_EDA,
    ef.eph_complete_index_EDA,
    ef.eph_complete_label_EDA,
    ef.is_total_cycle_downgrade_EDA,
    ef.home_vector_alignment_score,
    ef.pickup_ambiguity,
    ef.dropoff_ambiguity,
    ef.day_of_week,
    ef.time_of_day_block,
    ef.day_type,

    -- 3. GEOSPATIAL: table 'silver_palette' (sp)
    sp.dropoff_polygon_id,
    sp.dropoff_polygon_name,
    sp.dropoff_h3_hex_id,
    sp.dropoff_hdbscan_id,
    sp.dropoff_hdbscan_name,
    sp.realized_traffic_index,
    sp.historical_rolling_avg_traffic_index,
    sp.traffic_volatility_index_ml,
    sp.traffic_volatility_index_eda,

    -- 4. CONTEXT: table 'heuristic_flag' (hf) via 'heuristic_flag_offers' (hfo)
    hf.heuristic_flag_description AS heuristic_flag_context

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef ON o.offer_id = ef.offer_id_fk
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.silver_palette` sp ON o.offer_id = sp.offer_id
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.heuristic_flag_offers` hfo ON o.offer_id = hfo.offers_offer_id
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.heuristic_flag` hf ON hfo.heuristic_flag_heuristic_flag_id = hf.heuristic_flag_id
"""

try:
    print(f"Building full view: {view_name}...")
    client.query(sql_logic).result()
    print(f"SUCCESS: {view_name} rebuilt with 100% column parity (including hour_of_day).")
except Exception as e:
    print(f"FAILURE in {view_name}: {e}")

Building full view: v_ML_Supervised...
SUCCESS: v_ML_Supervised rebuilt with 100% column parity (including hour_of_day).


## Phase 5 — Parity audit

Certifies row and column parity between the SQLite source and the BigQuery destination.

#### 5.1 — SQLite vs. BigQuery parity audit

Compares row and column counts for every table and view across both systems and flags any mismatch.

In [15]:
def perform_parity_audit():
    print("Starting structural parity audit...")
    print(f"   Source: {DB_PATH}")
    print(f"   Destination: {PROJECT_ID}.{DATASET_ID}\n")

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT name, type FROM sqlite_master WHERE name NOT LIKE 'sqlite_%';")
    local_objects = cursor.fetchall()

    sqlite_results = []
    for name, obj_type in local_objects:
        cursor.execute(f"PRAGMA table_info('{name}')")
        col_count = len(cursor.fetchall())
        cursor.execute(f"SELECT COUNT(*) FROM '{name}'")
        row_count = cursor.fetchone()[0]

        sqlite_results.append({
            'name': name.lower(),
            'type': obj_type.upper(),
            'sq_rows': row_count,
            'sq_cols': col_count
        })
    conn.close()
    df_sqlite = pd.DataFrame(sqlite_results)

    # BigQuery doesn't store view row counts in metadata, so query them directly
    bq_query = f"""
        SELECT
            table_name as name,
            table_type as type
        FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
    """
    df_bq_meta = client.query(bq_query).to_dataframe()

    bq_results = []
    for _, row in df_bq_meta.iterrows():
        name = row['name']
        obj_type = "TABLE" if row['type'] == "BASE TABLE" else "VIEW"

        table_ref = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{name}")
        col_count = len(table_ref.schema)

        row_query = f"SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.{name}`"
        row_count = client.query(row_query).to_dataframe().iloc[0,0]

        bq_results.append({
            'name': name.lower(),
            'bq_rows': row_count,
            'bq_cols': col_count
        })
    df_bq = pd.DataFrame(bq_results)

    audit = pd.merge(df_sqlite, df_bq, on='name', how='outer')

    audit['row_match'] = audit['sq_rows'] == audit['bq_rows']
    audit['col_match'] = audit['sq_cols'] == audit['bq_cols']

    def get_status(row):
        if pd.isna(row['bq_rows']): return "MISSING IN BQ"
        if row['row_match'] and row['col_match']: return "MATCH"
        res = ""
        if not row['row_match']: res += f"Row delta ({row['bq_rows'] - row['sq_rows']}) "
        if not row['col_match']: res += f"Col delta ({row['bq_cols'] - row['sq_cols']}) "
        return "MISMATCH: " + res

    audit['status'] = audit.apply(get_status, axis=1)

    print(audit[['name', 'type', 'sq_rows', 'bq_rows', 'sq_cols', 'bq_cols', 'status']].to_string(index=False))

    mismatches = audit[audit['status'] != "MATCH"].shape[0]
    if mismatches == 0:
        print("\nPARITY CERTIFICATION: SUCCESSFUL. The star schema is 1:1.")
    else:
        print(f"\nATTENTION: {mismatches} discrepancies detected. Review log.")

perform_parity_audit()

Starting structural parity audit...
   Source: /workspaces/pienza/data/pienza.db
   Destination: 645009831643.pienza_mini

                      name  type  sq_rows  bq_rows  sq_cols  bq_cols                   status
         activity_earnings TABLE     3446     3446        7        7                    MATCH
   driver_state_at_request TABLE        2        2        2        2                    MATCH
       engineered_features TABLE     4765     4765       45       45                    MATCH
               event_types TABLE        5        5        3        3                    MATCH
            heuristic_flag TABLE        8        8        2        2                    MATCH
     heuristic_flag_offers TABLE      831      831        2        2                    MATCH
     interpolation_quality TABLE        5        5        2        2                    MATCH
            lifetime_trips TABLE     3446     3446       15       15                    MATCH
              offer_action TABL

<div style="border-left: 4px solid #21918c; padding: 4px 20px; background: rgba(33,145,140,0.05); border-radius: 0 6px 6px 0;">

_Ignore the noise above from unrelated columns — the `v_ML_Supervised` mismatch is expected, since one column (`hour_of_day`) was added after the SQLite baseline was captured._

</div>

## Phase 6 — Backup

Ensures the raw SQLite file itself is also preserved in GCS as a fallback.

#### 6.1 — Conditional GCS backup

Checks whether the master .db file already exists in the GCS bucket before uploading, to avoid redundant transfers.

In [16]:
from google.cloud import storage
import os

def investigate_and_seal():
    print("Checking backup destination...")
    print("-" * 65)

    BUCKET_NAME = 'pienza-streamlit'
    GCS_DESTINATION_PATH = f'backups/{os.path.basename(DB_PATH)}'

    try:
        storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)

        bucket = storage_client.bucket(BUCKET_NAME)
        blob = bucket.blob(GCS_DESTINATION_PATH)

        print(f"Looking for '{GCS_DESTINATION_PATH}' in bucket '{BUCKET_NAME}'...")

        if blob.exists():
            print("\nFOUND: The file already exists in the cloud.")
            print("SKIPPED: No re-upload needed.")
        else:
            print(f"\nRESULT: The file was not found in '{BUCKET_NAME}'.")
            print(f"Starting upload of: {os.path.basename(DB_PATH)}...")

            blob.upload_from_filename(DB_PATH)

            print(f"SUCCESS: Uploaded to gs://{BUCKET_NAME}/{GCS_DESTINATION_PATH}")

        print("-" * 65)
        print("Backup protocol finished.")

    except Exception as e:
        print(f"FAILURE during backup check: {e}")

investigate_and_seal()

Checking backup destination...
-----------------------------------------------------------------
Looking for 'backups/pienza.db' in bucket 'pienza-streamlit'...

FOUND: The file already exists in the cloud.
SKIPPED: No re-upload needed.
-----------------------------------------------------------------
Backup protocol finished.


---

## Appendix — variable journey

```mermaid
flowchart TD
    sqlite["pienza.db (SQLite)"] --> engine["db_engine\n(SQLAlchemy)"]
    sqlite --> client["client\n(BigQuery, via prepare_dataset)"]

    sqlite --> migration["complete_migration()\n(every table -> BQ native table)"]
    migration --> audit["execute_relational_audit()\n(FK orphan/connectivity checks)"]

    sqlite --> extract["extract_view_logic()\n(reference: original SQLite view SQL)"]

    migration --> funnel_m["v_trip_funnel_metrics"]
    migration --> funnel_w["v_trip_funnel_wide"]
    funnel_w --> kpis["v_trip_final_kpis"]
    kpis --> dossier["v_mission_dossier"]

    migration --> broche["v_broche_fks\n(full lineage audit)"]
    migration --> offers_human["v_offers_human\n(dimension labels resolved)"]

    migration --> lifecycle["v_lifecycle_audit"]
    lifecycle --> lifecycle_accepted["v_lifecycle_audit_accepted"]

    migration --> ml_supervised["v_ML_Supervised\n(offers + engineered_features +\nsilver_palette + heuristic flags)"]

    dossier --> parity["perform_parity_audit()\n(row/column parity, SQLite vs. BQ)"]
    broche --> parity
    offers_human --> parity
    lifecycle_accepted --> parity
    ml_supervised --> parity

    sqlite --> backup["investigate_and_seal()\n(raw .db file -> GCS backup)"]

    classDef default stroke:#21918c,stroke-width:2px;
    linkStyle default stroke:#21918c,stroke-width:2px;
```